## Just imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import justNRE
# import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score

from tqdm import tqdm

import ultranest
from ultranest import ReactiveNestedSampler
from ultranest.stepsampler import SliceSampler

from scipy.interpolate import interp1d
import pandas as pd


## Load parameter and data arrays

Specify data and parameters in shape `(numbers_of_simulations, data/paramater dimensionallity)` and specify the names of the parameters.

In [ ]:
# Load a mock example
data_arr, params_arr = justNRE.generate_mock_raw_data(); print('Shape of data:', data_arr.shape, 'Shape of parameters:', params_arr.shape)

param_names = ['param']

n_params = params_arr.shape[1]

# Track the true param range so we can set a sane prior later (see Inference section)
param_low, param_high = params_arr.min(axis=0), params_arr.max(axis=0)


## Train an ML algorithm

Here we train an ensemble of networks and take their average for the final result. This is not strictly necessary but it leads to more stable results.

In [ ]:
instances, targets = justNRE.prepare_for_NRE(data_arr, params_arr)

(X_train, y_train), (X_val, y_val), (X_test, y_test) = justNRE.split(instances, targets, [80, 10, 10])

(X_train_norm, X_val_norm, X_test_norm), denormalize, normalize = justNRE.normalize((X_train, X_val, X_test), norm_type='standard')


In [ ]:
# Network / training hyperparameters -- passed explicitly to train_nre_ensemble
# instead of being read as module-level globals.
nre_ensemble, loss_figs = justNRE.train_nre_ensemble(
    X_train_norm, y_train, X_val_norm, y_val,
    num_models=2,          # e.g. 5 for a sturdier ensemble
    base_seed=42,
    epochs=150,
    batch_size=128,
    patience=40,
    dropout=0.3,
    lr=1e-3,
    loss='binary_crossentropy',
    last_activation='sigmoid',
    verbose=0,              # silent per-epoch training
    plot=True,
)

# Show each ensemble member's train/val loss curve
for fig in loss_figs:
    fig.show()


In [ ]:
y_proba = justNRE.ensemble_predict(nre_ensemble, X_test, normalize_fn=normalize)
print("ROC AUC:", roc_auc_score(y_test, y_proba))


In [ ]:
fig, conf_matrix = justNRE.plot_confusion_matrix(X_val_norm, y_val, nre_ensemble[0])
fig.show()
print(conf_matrix)


# Inference

Choose a mock observation from the test set

In [ ]:
# Choose one
which_one = 13

obs_true   = np.array(X_test[which_one, :-n_params])
theta_true = np.array(X_test[which_one, -n_params:]); print(theta_true)


Specify a prior transform.

**Note:** the prior bounds below are set to the true range of the *training* parameters (`param_low`/`param_high`), not a fixed `[0, 1]`. The classifier only ever saw joint examples with `theta` in that range, so evaluating it outside that range means extrapolating -- the log-ratio estimate is not reliable there. If you have real (non-mock) data, set these bounds to match your simulator's actual prior range.

In [ ]:
l_bounds = param_low
u_bounds = param_high

prior_transform = lambda cube: cube * (u_bounds - l_bounds) + l_bounds


In [ ]:
def loglike_vector(params):
    return justNRE.log_ratio_vectorized(obs_true, params, nre_ensemble, normalize)


In [ ]:
sampler = ReactiveNestedSampler(param_names, loglike_vector, transform=prior_transform, vectorized=True)
result = sampler.run(max_ncalls=1_000_000, dlogz=0.5)  # Adjust for convergence

print("Best-fit parameters:", result['maximum_likelihood']['point'])
sampler.print_results()
post_samples = np.array(result['samples'])


In [ ]:
fig = plt.figure(figsize=(n_params*1.5+2, n_params*1.5+2))

fig = justNRE.cornerplot1(post_samples, theta_true, param_names, fig, "lightblue")
